# Week 1: Probing Pretrained Models

## Overview
This notebook is the first hands-on exercise in the Applied AI Engineering Fellowship. The goal is to build direct, empirical intuition for how a pretrained transformer language model actually behaves at the input (tokenization) and output (next-token prediction) boundaries.

## Concepts covered
- **Tokenization**: the process of converting raw text into a sequence of integer IDs a model can consume. Modern LLMs use subword tokenization (commonly Byte-Pair Encoding, BPE), which breaks text into frequently-occurring chunks rather than whole words or single characters. This is why the same string can be split differently -- sometimes surprisingly -- by different tokenizers.
- **Embeddings**: each token ID is mapped to a dense vector via a lookup table (the embedding matrix). This vector is the model's initial representation of that token before any contextual processing happens.
- **Self-attention**: the mechanism, introduced in the paper we're reproducing, that lets each token's representation be updated by attending to (weighting and combining information from) every other token in the sequence. It's what gives transformers context-sensitivity without recurrence.
- **Autoregressive generation**: the model produces text one token at a time, each new token conditioned on all previous ones. At each step it outputs a probability distribution over the entire vocabulary for "what comes next," and generation is the repeated process of sampling (or picking) from that distribution.

## Task
Probe a pretrained model's tokenizer edge cases and next-token distributions across prompts, using two models for comparison.

## Setup
Models used: GPT-2 (124M) and Qwen2.5-0.5B. Both are freely available, no access request required.

## Reproduction thread
Ties to core claims from *Attention Is All You Need* and *Language Models are Few-Shot Learners* -- specifically how tokenization choices and next-token prediction behave in practice, not just on paper.

## Hypothesis and Observation
Each comparison below has a **hypothesis** cell before the code and an **observation** cell after it. Write your prediction before running the cell, then fill in what actually happened after running it in Colab. Predicting first -- and sometimes being wrong -- is the point: it's how you calibrate your mental model of how these systems actually work.

In [ ]:
# Install dependences
!pip install -q transformers accelerate torch

### Loading the models
This cell loads two tokenizer/model pairs from Hugging Face: GPT-2 (2019, 124M parameters, a BPE tokenizer trained on WebText) and Qwen2.5-0.5B (2024, 0.5B parameters, a BPE tokenizer trained on a much larger and more multilingual corpus). It runs on Colab's free GPU if available, otherwise falls back to CPU. "Loading a model" here means downloading its weights and instantiating both the tokenizer (a fixed vocabulary plus merge rules) and the neural network itself in evaluation mode (`.eval()` disables dropout and other training-only behavior).

In [ ]:
import os
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load secret and set as HF_TOKEN
os.environ["HF_TOKEN"] = userdata.get("Flow_HF_Token")

MODEL_IDS = {
    "gpt2": "gpt2",
    "qwen": "Qwen/Qwen2.5-0.5B",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizers = {name: AutoTokenizer.from_pretrained(mid) for name, mid in MODEL_IDS.items()}
models = {
    name: AutoModelForCausalLM.from_pretrained(mid, dtype=torch.float32).to(device).eval()
    for name, mid in MODEL_IDS.items()
}

print(f"Loaded {list(MODEL_IDS.values())} on {device}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loaded ['gpt2', 'Qwen/Qwen2.5-0.5B'] on cuda


Hypothesis: tokenizer edge cases

**Quick context:** GPT-2 uses byte-level BPE trained on roughly 40GB of web text (WebText) with a 50,257-token vocabulary. Qwen2.5 uses a larger BPE vocabulary (~151K tokens) trained on a much bigger, more multilingual, and more code-heavy corpus. A smaller vocabulary generally forces more strings into more, smaller pieces; a larger, more diverse vocabulary tends to have more whole-word or whole-phrase tokens available, especially outside English.

Before running the next cell, predict which model's tokenizer will split each string into more tokens, and why.

Numbers (3.14159):
Whitespace runs (a    b):
Non-English text (konnichiwa in Japanese script):
Made-up word (flibbertigibbet):
Code snippet (def foo(): return x+1):

### Running the tokenizer comparison
This cell feeds five deliberately tricky strings -- a decimal number, a run of repeated whitespace, non-English (Japanese) text, an invented word, and a short code snippet -- through both tokenizers. For each, it prints the number of tokens produced and the decoded text of each individual token, so you can see exactly where each tokenizer chose to cut the string. Comparing token counts and boundaries side by side is one of the fastest ways to build intuition for how vocabulary size and training data shape a tokenizer's behavior on inputs it wasn't optimized for.

In [ ]:
edge_cases = [
    "3.14159",
    "a    b",
    "\u3053\u3093\u306b\u3061\u306f",
    "flibbertigibbet",
    "def foo():\n    return x+1",
]

for text in edge_cases:
    print(f"Input: {text!r}")
    for name, tok in tokenizers.items():
        ids = tok.encode(text)
        pieces = [tok.decode([i]) for i in ids]
        print(f"  {name}: {len(ids)} tokens -> {pieces}")
    print()

Input: '3.14159'
  gpt2: 4 tokens -> ['3', '.', '14', '159']
  qwen: 7 tokens -> ['3', '.', '1', '4', '1', '5', '9']

Input: 'a    b'
  gpt2: 5 tokens -> ['a', ' ', ' ', ' ', ' b']
  qwen: 3 tokens -> ['a', '   ', ' b']

Input: 'こんにちは'
  gpt2: 6 tokens -> ['こ', 'ん', 'に', '�', '�', 'は']
  qwen: 1 tokens -> ['こんにちは']

Input: 'flibbertigibbet'
  gpt2: 6 tokens -> ['fl', 'ib', 'bert', 'ig', 'ib', 'bet']
  qwen: 6 tokens -> ['fl', 'ib', 'bert', 'ig', 'ib', 'bet']

Input: 'def foo():\n    return x+1'
  gpt2: 11 tokens -> ['def', ' foo', '():', '\n', ' ', ' ', ' ', ' return', ' x', '+', '1']
  qwen: 8 tokens -> ['def', ' foo', '():\n', '   ', ' return', ' x', '+', '1']



Observation: tokenizer edge cases

Qwen2.5 tokenized every string into the same number of tokens as GPT-2 or fewer across all five cases, which is the clearest pattern in this data:

- **Numbers (3.14159):** GPT-2 used 4 tokens (`3`, `.`, `14`, `159`), while Qwen split into 7, essentially one token per digit. GPT-2's BPE merges happened to have learned merges for `14` and `159` from its training data; Qwen's larger, more diverse vocabulary apparently didn't preserve those specific digit-group merges, so it fell back to near character-level splitting for this number.
- **Whitespace runs (`a    b`):** GPT-2 split the four spaces into four separate single-space tokens plus `a` and `' b'` (5 tokens total). Qwen collapsed the run into a single 3-space token (3 tokens total). This is a direct, visible consequence of Qwen's tokenizer having learned multi-space run tokens -- likely from code-heavy training data, where indentation runs are common.
- **Non-English text (konnichiwa):** this is the largest gap in the whole comparison. GPT-2, trained on English-dominant WebText, has no dedicated tokens for Japanese characters, so it fell back to byte-level fragments -- 6 tokens, two of which decode as broken replacement-character fragments because a single Japanese character got split across multiple raw bytes. Qwen encoded the entire five-character greeting as a single token, reflecting its explicitly multilingual training data and larger vocabulary.
- **Made-up word (flibbertigibbet):** both models split it identically into the same 6 subword pieces (`fl`, `ib`, `bert`, `ig`, `ib`, `bet`). For a genuinely novel English string built from common English subword fragments, vocabulary size stopped mattering -- both tokenizers decomposed it the same way.
- **Code snippet:** GPT-2 needed 11 tokens, splitting the indentation into three separate single-space tokens; Qwen needed 8, again collapsing the indentation run and merging `():\n` into a single token. Consistent with the whitespace-run result, this suggests Qwen's training corpus included enough source code that common code idioms (indentation, `():`) earned their own tokens.

**Net pattern:** the gap between the two tokenizers was largest exactly where GPT-2's training data was thinnest -- non-English text and structured whitespace like code indentation -- and it vanished on a novel-but-English word built from ordinary subwords. This is consistent with vocabulary size and training-data composition, rather than tokenizer algorithm (both use BPE), driving most of the difference.

Hypothesis: next-token distributions

**Quick context:** entropy here measures how spread out the model's probability distribution over the next token is. Low entropy means the model puts most of its probability mass on a small number of candidates (it's "confident" about what comes next); high entropy means probability is spread across many plausible continuations (the model is more "uncertain"). A factual prompt with one clearly correct completion should, in principle, produce lower entropy than an open-ended or nonsensical one -- that's the hypothesis to test below.

Before running the next cells, predict which model will be more confident (lower entropy) for each prompt category, and why.

Factual (The capital of France is):
Ambiguous (My favorite thing about weekends is):
Adversarial or nonsensical (The purple triangle whispered):

### Defining the entropy / top-k helper
This function runs a single forward pass of the model on a prompt and reads off the logits (raw, unnormalized scores) at the final token position -- i.e. what the model predicts should come next. It converts those logits into a probability distribution with softmax, computes the Shannon entropy of that distribution (in nats, since we use the natural log) as a scalar measure of the model's uncertainty, and returns the top-10 candidate next tokens with their probabilities so you can inspect not just how confident the model is, but what it's confident about.

In [ ]:
import torch.nn.functional as F

def next_token_distribution(model, tokenizer, prompt, k=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k)
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
    top_k = [(tokenizer.decode([i]), round(p.item(), 4)) for i, p in zip(top_ids, top_probs)]
    return entropy, top_k

### Running the comparison across prompt categories
This cell runs the helper above across both models and three prompt categories chosen to probe different regimes: a **factual** prompt with a single correct answer, an **ambiguous** open-ended prompt with many reasonable completions, and an **adversarial / nonsensical** prompt with no clear "correct" continuation at all. For each combination it prints the entropy and top-10 next-token candidates, so you can compare both the raw uncertainty number and the actual words each model considered plausible.

In [ ]:
prompts = {
    "factual": "The capital of France is",
    "ambiguous": "My favorite thing about weekends is",
    "adversarial": "The purple triangle whispered",
}

for label, prompt in prompts.items():
    print(f"Prompt category: {label} -> {prompt!r}")
    for name in models:
        entropy, top_k = next_token_distribution(models[name], tokenizers[name], prompt)
        print(f"  {name}: entropy={entropy:.3f}")
        print(f"    top-10: {top_k}")
    print()

Prompt category: factual -> 'The capital of France is'
  gpt2: entropy=5.998
    top-10: [(' the', 0.0846), (' now', 0.0479), (' a', 0.0462), (' France', 0.0324), (' Paris', 0.0322), (' in', 0.0266), (' also', 0.0264), (' not', 0.0238), (' home', 0.0233), (' still', 0.0155)]
  qwen: entropy=3.258
    top-10: [(' Paris', 0.3156), (' ______', 0.1146), (' ____', 0.0635), (' __', 0.0551), (':\n', 0.0513), (' located', 0.0359), (' the', 0.028), ('\n', 0.0246), (' (', 0.0197), (' .\n', 0.0188)]

Prompt category: ambiguous -> 'My favorite thing about weekends is'
  gpt2: entropy=3.754
    top-10: [(' that', 0.2477), (' the', 0.1355), (' when', 0.0674), (' you', 0.0415), (' being', 0.0307), (' how', 0.0285), (' having', 0.0269), (' I', 0.0261), (' getting', 0.0245), (' we', 0.02)]
  qwen: entropy=4.799
    top-10: [(' the', 0.1301), (' that', 0.1029), (' getting', 0.0549), (' when', 0.047), (' going', 0.0436), (' being', 0.0324), (' spending', 0.0229), (' having', 0.0193), (' watching', 0.0187

Observation: next-token distributions

The results didn't fully match a simple "factual prompt = lowest entropy" story -- the ranking between models actually flipped depending on the category:

- **Factual prompt:** entropy dropped sharply from GPT-2 (5.998) to Qwen (3.258), roughly a 46% reduction, and Qwen's top prediction (' Paris') carried 31.6% of the probability mass. GPT-2's top token was the generic ' the' at just 8.5%, with ' Paris' only its 5th-ranked candidate at 3.2%. Qwen clearly "knows" this fact with far more confidence than GPT-2 does, consistent with Qwen being a newer, more capable model despite only being about 4x larger.
- **Ambiguous prompt:** here the ranking reversed -- GPT-2 was more confident (entropy 3.754) than Qwen (entropy 4.799). Both models' top candidates were generic continuations (' that', ' the'), but GPT-2 concentrated more probability on its top choice (24.8%) than Qwen did on its own (13.0%), and Qwen's top-10 list was semantically more varied (' watching', ' eating', ' spending' vs. GPT-2's more function-word-heavy list). On a genuinely open-ended prompt, the larger/newer model was actually less peaked, not more.
- **Adversarial / nonsensical prompt:** GPT-2 was again more confident (4.492) than Qwen (5.134), though both had noticeably higher entropy than either model showed on the factual prompt, and neither model's top candidates suggested any special "confusion" -- both simply fell back to generic connective tokens (' to', ' in', ',').
- **Consistency check:** the gap between models was not consistent across categories. Qwen was much more confident than GPT-2 on the factual prompt but slightly less confident than GPT-2 on both the ambiguous and adversarial prompts. This suggests low entropy isn't a stable per-model trait of "confidence" so much as a signal tied to whether the model has a strong, specific, memorized answer available -- which was only clearly true for the factual prompt, where a single named fact (Paris) has an unambiguous target.

**Takeaway:** entropy tracked something closer to "does this model have a specific fact to retrieve" rather than a general per-model confidence level. That's worth carrying forward: entropy alone can't tell you *why* a model is confident, only *that* it is -- you have to inspect the top-k tokens themselves to see whether that confidence is grounded in something like factual recall.